# 04b — Social Media Charts (Altair)

Publication-ready charts using the @unwelcomedata brand palette (Coolors).
These render inline for preview and export to `outputs/` as exact-dimension PNGs.

Charts:
1. Scatter — consumption vs fatality rate (by region)
2. Ranked bars — top/bottom 10 states (per VMT)
3. Comparison — IID vs non-IID
4. Trend — national fatalities 2015–2020
5. Choropleth — fatality rate per 100M VMT
6. Choropleth — felony status (category)

In [ ]:
import sys
import os
from pathlib import Path

import importlib
import pandas as pd
import yaml

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import src.viz_social
importlib.reload(src.viz_social)
from src.viz_social import (
    social_scatter,
    social_ranked_bars,
    social_diverging_bars,
    social_residual_bars,
    social_comparison,
    social_trend,
    social_choropleth,
    social_bubble_choropleth,
    social_bivariate_choropleth,
    save_social,
)

with open(PROJECT / "config.yaml") as f:
    cfg = yaml.safe_load(f)

df = pd.read_parquet(PROJECT / "export" / "dui_by_state_v2.parquet")
print(f"Project: {PROJECT.name}")
print(f"Master table: {df.shape[0]} states x {df.shape[1]} columns")

## 1. Scatter — Consumption vs Fatality Rate

In [ ]:
chart = social_scatter(
    df,
    x="ethanol_per_capita_gallons_2022",
    y="alcohol_fatality_rate_per_100m_vmt",
    color_by="region",
    title="Alcohol Consumption vs Impaired-Driving Deaths",
    subtitle="Per-VMT fatality rate controls for driving exposure. Each dot is a state.",
    source="NIAAA 2022, NHTSA FARS 2024, FHWA VMT 2022",
    xlabel="Per capita ethanol (gallons, 2022)",
    ylabel="Alcohol fatalities per 100M VMT",
    label_col="state_abbr",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_scatter_consumption_vs_fatality", preset="twitter_landscape")
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_scatter_consumption_vs_fatality.png")))


## 2. Ranked Bars — Top/Bottom 10 (per VMT)

In [ ]:
chart = social_ranked_bars(
    df,
    x="state_name",
    y="alcohol_fatality_rate_per_100m_vmt",
    title="Worst & Best States for Impaired-Driving Deaths",
    subtitle="Alcohol fatalities per 100M vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    top_n=10,
    bottom_n=10,
    preset="instagram_portrait",
)
save_social(chart, cfg, "social_ranked_top_bottom_10", preset="instagram_portrait")
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_ranked_top_bottom_10.png")))


## 3. IID vs Non-IID Comparison

In [ ]:
df["iid_group"] = df["iid_all_offender"].map({1: "IID for all offenders", 0: "No universal IID"})

chart = social_comparison(
    df,
    group_col="iid_group",
    value_col="alcohol_fatality_rate_per_100m_vmt",
    title="Does Mandatory IID Reduce Impaired-Driving Deaths?",
    subtitle="Mean alcohol fatality rate per 100M VMT by IID policy",
    source="NHTSA FARS 2024, FHWA VMT 2022, IIHS/GHSA",
    colors={"IID for all offenders": "#2A9D8F", "No universal IID": "#E76F51"},
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_comparison_iid", preset="twitter_landscape")
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_comparison_iid.png")))


## 5. Choropleth — Fatality Rate per 100M VMT

In [ ]:
chart = social_choropleth(
    df,
    column="alcohol_fatality_rate_per_100m_vmt",
    title="Alcohol-Impaired Fatality Rate by State",
    subtitle="Deaths per 100 million vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    mode="heat",
    legend_title="Deaths per 100M VMT",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_map_fatality_rate_vmt", preset="twitter_landscape")
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_map_fatality_rate_vmt.png")))


## 6. Choropleth — Felony Status (category)

In [ ]:
df["felony_label"] = df["first_offense_felony"].map({1.0: "Can be felony", 0.0: "Always misdemeanor"})
df.loc[df["felony_label"].isna(), "felony_label"] = "Unknown"

chart = social_choropleth(
    df,
    column="felony_label",
    title="First-Offense DUI: Felony Possible?",
    subtitle="States where first DUI can be charged as felony",
    source="NCSL DUI/DWI criminal status laws",
    mode="category",
    category_colors={"Can be felony": "#E76F51", "Always misdemeanor": "#2A9D8F", "Unknown": "#E5E7EB"},
    legend_title="First-offense status",
    preset="twitter_landscape",
)
save_social(chart, cfg, "social_map_felony_status", preset="twitter_landscape")
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_map_felony_status.png")))


## Quick exploration

One-liner functions to explore any columns interactively. Just call and view.

In [ ]:
def quick_map(column, title=None, mode="heat", **kwargs):
    """Choropleth any column. mode='heat' for numeric, 'category' for discrete."""
    title = title or column.replace('_', ' ').title()
    return social_choropleth(df, column=column, title=title, mode=mode, preset='twitter_landscape', **kwargs)


def quick_scatter(x, y, title=None, color_by='region', **kwargs):
    """Scatter any two numeric columns. Colored by region by default."""
    title = title or f"{x.replace('_',' ').title()} vs {y.replace('_',' ').title()}"
    return social_scatter(df, x=x, y=y, color_by=color_by, title=title, preset='twitter_landscape', **kwargs)


def quick_bars(y, x='state_name', title=None, top_n=10, bottom_n=0, **kwargs):
    """Ranked bar chart. Shows top_n (and optionally bottom_n) states."""
    title = title or f"Top {top_n} States by {y.replace('_',' ').title()}"
    return social_ranked_bars(df, x=x, y=y, title=title, top_n=top_n, bottom_n=bottom_n, preset='instagram_portrait', **kwargs)


def quick_compare(group_col, value_col, title=None, **kwargs):
    """Compare group means for any grouping column vs any numeric column."""
    title = title or f"{value_col.replace('_',' ').title()} by {group_col.replace('_',' ').title()}"
    return social_comparison(df, group_col=group_col, value_col=value_col, title=title, preset='twitter_landscape', **kwargs)


def quick_trend(x='year', y='impaired_fatalities', data=None, title=None, **kwargs):
    """Line trend chart. Pass a different DataFrame via data= if needed."""
    title = title or f"{y.replace('_',' ').title()} over {x.replace('_',' ').title()}"
    src = data if data is not None else df
    return social_trend(src, x=x, y=y, title=title, preset='twitter_landscape', **kwargs)


def quick_bubble_map(color_col, size_col, title=None, **kwargs):
    """Choropleth with fill color + bubble overlay showing two variables."""
    title = title or f"{color_col.replace('_',' ').title()} (color) + {size_col.replace('_',' ').title()} (size)"
    return social_bubble_choropleth(df, color_col=color_col, size_col=size_col, title=title, preset='twitter_landscape', **kwargs)


def quick_bivariate_map(x_col, y_col, title=None, **kwargs):
    """Bivariate choropleth — two variables encoded as a 3x3 color grid.\n    Pass footnote='...' for methodology caveats."""
    title = title or f"{x_col.replace('_',' ').title()} vs {y_col.replace('_',' ').title()}"
    return social_bivariate_choropleth(df, x_col=x_col, y_col=y_col, title=title, preset='twitter_landscape', **kwargs)


def quick_diverging(value_col, expected='mean', title=None, top_n=15, **kwargs):
    """Diverging bar chart — bars extend left (better) or right (worse) from expected."""
    title = title or f"{value_col.replace('_',' ').title()}: Better vs Worse Than Expected"
    return social_diverging_bars(df, value_col=value_col, expected=expected, title=title, top_n=top_n, preset='instagram_portrait', **kwargs)


def quick_residual_bars(outcome_col, predictor_col, title=None, top_n=15, **kwargs):
    """Diverging bars of regression residuals — actual vs predicted from one predictor."""
    title = title or f"{outcome_col.replace('_',' ').title()}: Residual from {predictor_col.replace('_',' ').title()}"
    return social_residual_bars(df, outcome_col=outcome_col, predictor_col=predictor_col, title=title, top_n=top_n, preset='instagram_portrait', **kwargs)


print('Quick functions ready: quick_map, quick_scatter, quick_bars, quick_compare, quick_trend, quick_bubble_map, quick_bivariate_map, quick_diverging, quick_residual_bars')

In [ ]:
def cols():
    """Print all columns in the master table, grouped by type."""
    numeric = [c for c in df.columns if df[c].dtype in ('float64', 'int64')]
    categorical = [c for c in df.columns if df[c].dtype == 'object']
    print(f'=== NUMERIC ({len(numeric)}) ===')
    for c in numeric:
        print(f'  {c}')
    print(f'\n=== CATEGORICAL ({len(categorical)}) ===')
    for c in categorical:
        vals = df[c].nunique()
        print(f'  {c}  ({vals} unique)')

cols()

In [ ]:
# --- Try it: consumption vs arrest rate ---
quick_scatter('ethanol_per_capita_gallons_2022', 'dui_arrest_rate_per_100k_reporting',
              title='Alcohol Consumption vs DUI Arrest Rate')

In [ ]:
# More examples (uncomment any):
# quick_map('dui_arrest_rate_per_100k_reporting', title='DUI Arrest Rate per 100k')
# quick_map('ethanol_per_capita_gallons_2022', title='Per Capita Alcohol Consumption')
# quick_scatter('dui_arrest_rate_per_100k_reporting', 'alcohol_fatality_rate_per_100m_vmt')
# quick_bars('dui_arrest_rate_per_100k_reporting', top_n=10, bottom_n=10)
# quick_compare('iid_group', 'dui_arrest_rate_per_100k_reporting')
# quick_map('pct_suspended', title='% Drivers on Suspended License in Fatal Crashes')

## 7. Diverging Bars — Better/Worse Than Expected

Which states perform better or worse than the national average (or model prediction)?

In [ ]:
# Simple divergence from the mean per-VMT fatality rate
chart = social_diverging_bars(
    df,
    value_col='alcohol_fatality_rate_per_100m_vmt',
    label_col='state_name',
    expected='mean',
    title='Alcohol Fatality Rate: Who Is Above/Below Average?',
    subtitle='Deviation from mean per-VMT fatality rate across all states.',
    source='NHTSA FARS 2024, FHWA VMT 2022',
    footnote='Expected = national mean. Coral = worse than average. Teal = better.',
    top_n=12,
    preset='instagram_portrait',
)
save_social(chart, cfg, 'social_diverging_fatality_rate', preset='instagram_portrait')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_diverging_fatality_rate.png")))


In [ ]:
# Residual-based: worse/better than expected GIVEN consumption level
chart = social_residual_bars(
    df,
    outcome_col='alcohol_fatality_rate_per_100m_vmt',
    predictor_col='ethanol_per_capita_gallons_2022',
    label_col='state_name',
    title='More Deaths Than Drinking Explains?',
    subtitle='Residual: actual fatality rate minus what consumption alone predicts.',
    source='NIAAA 2022, NHTSA FARS 2024, FHWA VMT 2022',
    top_n=12,
    preset='instagram_portrait',
)
save_social(chart, cfg, 'social_residual_consumption_fatality', preset='instagram_portrait')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_residual_consumption_fatality.png")))


## 8. Regional Comparison — Total vs Alcohol Traffic Deaths

Does the regional pattern for alcohol fatalities mirror all-cause traffic deaths?
Or does alcohol disproportionately explain deaths in some regions?

In [ ]:
import numpy as np

# Aggregate by region (population-weighted rates)
regional = df.groupby('region', observed=True).agg(
    total_pop=('pop_2024', 'sum'),
    total_fatalities=('traffic_fatalities_2024', 'sum'),
    alcohol_fatalities=('alcohol_fatalities_nhtsa_imputed', 'sum'),
).reset_index()
regional['total_rate_per_100k'] = regional['total_fatalities'] / regional['total_pop'] * 100_000
regional['alcohol_rate_per_100k'] = regional['alcohol_fatalities'] / regional['total_pop'] * 100_000
regional['non_alcohol_rate_per_100k'] = regional['total_rate_per_100k'] - regional['alcohol_rate_per_100k']
regional['alcohol_share_pct'] = regional['alcohol_fatalities'] / regional['total_fatalities'] * 100

print(regional[['region', 'total_rate_per_100k', 'alcohol_rate_per_100k',
                'non_alcohol_rate_per_100k', 'alcohol_share_pct']].round(2).to_string(index=False))

In [ ]:
# Stacked bar: alcohol vs non-alcohol by region
import altair as alt
from viz import COOLORS, PALETTE, REGION_COLORS

# Melt for stacked chart
stacked = regional[['region', 'alcohol_rate_per_100k', 'non_alcohol_rate_per_100k']].melt(
    id_vars='region', var_name='cause', value_name='rate_per_100k'
)
stacked['cause'] = stacked['cause'].map({
    'alcohol_rate_per_100k': 'Alcohol-impaired',
    'non_alcohol_rate_per_100k': 'Non-alcohol'
})

# Order regions by total rate
region_order = regional.sort_values('total_rate_per_100k', ascending=False)['region'].tolist()

chart = alt.Chart(stacked).mark_bar(cornerRadiusEnd=3).encode(
    x=alt.X('region:N', sort=region_order, title='', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('rate_per_100k:Q', title='Fatalities per 100k population'),
    color=alt.Color('cause:N',
        scale=alt.Scale(domain=['Alcohol-impaired', 'Non-alcohol'],
                        range=[COOLORS['coral'], COOLORS['charcoal']]),
        legend=alt.Legend(title='Cause')),
    order=alt.Order('cause:N', sort='descending'),
).properties(
    width=500, height=350,
    title=alt.Title(
        'Traffic Fatalities by Region: Alcohol vs All Other Causes',
        subtitle=['Population-weighted rate per 100k. South leads on total deaths; West leads on alcohol share.'],
    ),
)

# Add total label on top
totals = regional[['region', 'total_rate_per_100k']].copy()
totals_labels = alt.Chart(totals).mark_text(
    dy=-8, fontSize=12, fontWeight='bold', color=PALETTE['dark']
).encode(
    x=alt.X('region:N', sort=region_order),
    y=alt.Y('total_rate_per_100k:Q'),
    text=alt.Text('total_rate_per_100k:Q', format='.1f'),
)

final = (chart + totals_labels)
save_social(final, cfg, 'social_regional_total_vs_alcohol', preset='twitter_landscape')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_regional_total_vs_alcohol.png")))


In [ ]:
# Grouped comparison: alcohol share by region
share_chart = alt.Chart(regional).mark_bar(cornerRadiusEnd=4, width=50).encode(
    x=alt.X('region:N', sort=region_order, title='', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('alcohol_share_pct:Q', title='% of traffic deaths that are alcohol-impaired',
            scale=alt.Scale(domain=[0, 40])),
    color=alt.Color('region:N',
        scale=alt.Scale(domain=list(REGION_COLORS.keys()), range=list(REGION_COLORS.values())),
        legend=None),
).properties(
    width=450, height=300,
    title=alt.Title(
        'Alcohol Share of Traffic Deaths by Region',
        subtitle=['West: highest proportion of traffic deaths involve alcohol.'],
    ),
)

# Share labels
share_labels = alt.Chart(regional).mark_text(
    dy=-8, fontSize=13, fontWeight='bold', color=PALETTE['dark']
).encode(
    x=alt.X('region:N', sort=region_order),
    y=alt.Y('alcohol_share_pct:Q'),
    text=alt.Text('alcohol_share_pct:Q', format='.1f'),
)

# National average line
nat_avg = df['alcohol_fatalities_nhtsa_imputed'].sum() / df['traffic_fatalities_2024'].sum() * 100
avg_rule = alt.Chart(pd.DataFrame([{'y': nat_avg}])).mark_rule(
    strokeDash=[4,2], strokeWidth=1.5, color=PALETTE['mid']
).encode(y='y:Q')
avg_label = alt.Chart(pd.DataFrame([{'y': nat_avg, 'text': f'National avg: {nat_avg:.1f}%'}])).mark_text(
    align='right', dx=-5, dy=-8, fontSize=10, color=PALETTE['mid'], fontStyle='italic'
).encode(y='y:Q', text='text:N')

final_share = (share_chart + share_labels + avg_rule + avg_label)
save_social(final_share, cfg, 'social_regional_alcohol_share', preset='twitter_landscape')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social_regional_alcohol_share.png")))


### Interpretation

Key finding: The South has by far the highest overall traffic fatality rate (≈14.2/100k vs 6.9 in the Northeast).
But alcohol's SHARE of those deaths is actually highest in the West (≈33%) — meaning the South's
excess deaths come disproportionately from non-alcohol causes (speed, rural roads, seatbelt non-use).

Social angle: "The South's traffic death problem isn't mainly about drinking — it's about everything else."

## Two-variable maps

Compare two variables on one map using either bubbles or bivariate coloring.

In [ ]:
# Option A: Bubble map — color = consumption, bubble size = % traffic deaths from alcohol
quick_bubble_map(
    'ethanol_per_capita_gallons_2022',
    'pct_alcohol_nhtsa_imputed',
    title='Alcohol Consumption (color) vs % Traffic Deaths from Alcohol (bubble)',
    color_legend='Per capita ethanol (gal)',
    size_legend='% traffic deaths alcohol',
)

In [ ]:
# Option B: Bivariate choropleth — 3x3 color grid
# Bottom-left = low consumption + low alcohol death share
# Top-right = high consumption + high alcohol death share
quick_bivariate_map(
    'ethanol_per_capita_gallons_2022',
    'pct_alcohol_nhtsa_imputed',
    title='Bivariate: Consumption vs % Traffic Deaths from Alcohol',
    subtitle='States in the top-right drink more AND lose more lives to alcohol-impaired driving.',
    source='NIAAA 2022, NHTSA FARS 2024',
    footnote='Each variable split into terciles. Color encodes position in a 3×3 grid.',
    x_label='Consumption →',
    y_label='% Deaths Alcohol →',
)